# Run a Single Coupled Example - Dynamic Wflow -> SFINCS

Run one coupled Wflow-SFINCS event: Wflow starts from the prepared native instate, generates SFINCS inflow hydrographs, then SFINCS consumes that dynamic handoff.

Stage Contract
- Requires: built coupled Wflow/SFINCS bases, Event Catalog, warmup forcing/states from `b_prepare_wflow_dynamic_handoff.ipynb`, and reviewed source artifacts.
- Produces: dynamic Wflow `sfincs_discharge.nc`, dynamic handoff QA/acceptance, one staged SFINCS scenario folder, forcing QA plots, optional SFINCS outputs, and post-run diagnostics.
- Next: review diagnostics or batch scenarios in `../05_create_scenarios.ipynb`.


In [ ]:
import json
import os
import warnings
from pathlib import Path

os.environ.pop("DEBUG", None)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/flood-rm-matplotlib")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")

import pandas as pd
from IPython.display import HTML, Video, display

# standard paths and readiness tables.
from wflow_runs.notebook import load_runtime
# event selection, SFINCS staging, and handoff readiness.
from sfincs_runs.scenarios import (
    plan_example,
    stage_inland_coupled_example_forcing,
)
# stage one SFINCS run from the Event Catalog.
from sfincs_runs.scenarios.event_forcing import run_model
# meteo forcing, discharge handoff, and acceptance gates.
from wflow_runs import ensure_dynamic_handoff

runtime = load_runtime(Path("../..").resolve(), wflow_domain_review_required=False)
location_root = runtime.location_root
repo_root = runtime.repo_root
config = runtime.config
paths = runtime.paths
events_root = runtime.events_root

## 1 - Select a catalog event and verify dynamic Wflow handoff

`plan_example` checks the static/base-model contract and selects the catalog event used by both Wflow and SFINCS.


In [ ]:
EVENT_ID = None  # None -> highest-RP catalog event; or e.g. "design_0398"
CATALOG_PATH = "data/event_catalog/catalog/probability_catalog.csv"

example_plan = plan_example(
    config,
    {"location_root": location_root},
    catalog_path=CATALOG_PATH,
    event_id=EVENT_ID,
)

plan_summary = pd.Series({
    "status": example_plan.status,
    "event_id": example_plan.event_id,
    "event_reference_time": example_plan.event_reference_time,
    "catalog_path": example_plan.catalog_path,
    "handoff_path": example_plan.handoff_path,
    "wflow_event_dir": example_plan.wflow_event_dir,
    "wflow_discharge_forcing": example_plan.wflow_discharge_forcing,
    "sfincs_scenario_dir": example_plan.sfincs_scenario_dir,
    "sfincs_dry_run_command": example_plan.sfincs_dry_run_command,
}, name="inland_coupled_example_plan")
display(plan_summary)

if example_plan.issues:
    print("Blocking issues (resolve upstream, then rerun):")
    for issue in example_plan.issues:
        print("  -", issue)
if example_plan.status != "ready":
    raise RuntimeError("Coupled example inputs are not ready; resolve the blocking issues above.")

event_id = example_plan.event_id
discharge_nc = events_root / event_id / "sfincs_discharge.nc"
event_precip_nc = events_root / event_id / "precip.nc"
acceptance_json = events_root / event_id / "sfincs_discharge.dynamic_handoff.json"

## Rerun Control


In [ ]:
rerun = True

## 2 - Configure SFINCS run

Run Wflow first from the prepared native `instate/instates.nc`. The accepted `sfincs_discharge.nc` produced here is the upstream boundary forcing that SFINCS consumes at its native `src` inflow points.


In [ ]:
run_dynamic_wflow_handoff = True
handoff_run = ensure_dynamic_handoff(
    config,
    location_root,
    event_id,
    catalog_path=CATALOG_PATH,
    rerun=rerun,
    run=run_dynamic_wflow_handoff,
)

display(handoff_run.summary)
if handoff_run.meteo_report is not None:
    display(handoff_run.meteo_report)
if handoff_run.handoff_report is not None:
    display(handoff_run.handoff_report)

handoff_acceptance = handoff_run.acceptance
display(handoff_acceptance)

In [ ]:
# Plot helper definitions used by this cell only.
import numpy as np
import xarray as xr


def plot_wflow_event_handoff(discharge_nc, *, precipitation_nc=None, event_label=None, figsize=(13, 4.5)):
    discharge_nc = Path(discharge_nc)
    if not discharge_nc.exists():
        raise FileNotFoundError(discharge_nc)

    with xr.open_dataset(discharge_nc) as opened:
        ds = opened.load()
    if "discharge" not in ds:
        raise ValueError(f"{discharge_nc} lacks discharge")
    discharge = ds["discharge"]
    if "time" not in discharge.dims:
        raise ValueError("discharge has no time dimension")

    has_precip = precipitation_nc is not None and Path(precipitation_nc).exists()
    fig, axes = plt.subplots(1, 3 if has_precip else 2, figsize=figsize, constrained_layout=True)
    map_ax, hydro_ax = (axes[0], axes[-1])

    source_dim = next((dim for dim in discharge.dims if dim != "time"), None)
    if source_dim is None:
        source_peak = np.asarray([float(discharge.max(skipna=True))])
        x = np.array([0.0])
        y = np.array([0.0])
        names = np.array(["total"])
    else:
        peak = discharge.max(dim="time", skipna=True)
        source_peak = np.asarray(peak.values, dtype=float).ravel()
        x = np.asarray(ds.coords.get("x", np.arange(source_peak.size)), dtype=float).ravel()
        y = np.asarray(ds.coords.get("y", np.zeros(source_peak.size)), dtype=float).ravel()
        names = np.asarray(ds.coords.get("name", np.arange(source_peak.size)), dtype=str).ravel()

    points = map_ax.scatter(
        x,
        y,
        c=source_peak,
        cmap="plasma",
        s=np.clip(source_peak / max(np.nanmax(source_peak), 1.0) * 180.0, 45.0, 180.0),
        edgecolors="black",
        linewidths=0.55,
    )
    fig.colorbar(points, ax=map_ax, shrink=0.78, label="Peak discharge [m3 s$^{-1}$]")
    for xi, yi, name in zip(x, y, names, strict=False):
        map_ax.annotate(str(name), (xi, yi), xytext=(4, 4), textcoords="offset points", fontsize=7)
    map_ax.set_title("Wflow handoff peaks")
    map_ax.set_aspect("equal", adjustable="datalim")
    map_ax.set_xlabel("x")
    map_ax.set_ylabel("y")

    if has_precip:
        precip_ax = axes[1]
        with xr.open_dataset(precipitation_nc) as precip_opened:
            precip_ds = precip_opened.load()
        precip_name = "precip" if "precip" in precip_ds else next(iter(precip_ds.data_vars))
        rainfall = precip_ds[precip_name]
        total_rainfall = rainfall.sum(dim="time", skipna=True) if "time" in rainfall.dims else rainfall
        total_rainfall.plot(
            ax=precip_ax,
            cmap="Blues",
            cbar_kwargs=dict(shrink=0.78, label="Storm total rainfall [mm]"),
        )
        precip_ax.set_title("Staged rainfall total")
        precip_ax.set_aspect("equal", adjustable="datalim")

    times = pd.DatetimeIndex(pd.to_datetime(discharge["time"].values))
    if source_dim is None:
        hydro_ax.plot(times, np.asarray(discharge.values, dtype=float), color="#225ea8", linewidth=1.8)
    else:
        frame = discharge.transpose(source_dim, "time").to_pandas().T
        if len(names) == len(frame.columns):
            frame.columns = names
        for column in frame.columns:
            hydro_ax.plot(times, frame[column].to_numpy(dtype=float), linewidth=1.25, alpha=0.86)
    hydro_ax.set_title("Handoff hydrographs")
    hydro_ax.set_xlabel("time")
    hydro_ax.set_ylabel("discharge [m3 s$^{-1}$]")
    hydro_ax.grid(True, color="#d9d9d9", linewidth=0.7, alpha=0.8)
    hydro_ax.tick_params(axis="x", labelrotation=30)
    fig.suptitle(f"{event_label or discharge_nc.parent.name} Wflow dynamic handoff", y=1.03)
    return fig, axes

precip_for_plot = event_precip_nc if event_precip_nc.exists() else None
fig, axes = plot_wflow_event_handoff(
    discharge_nc,
    precipitation_nc=precip_for_plot,
    event_label=event_id,
)

## 3 - Stage SFINCS and apply rainfall + dynamic Wflow discharge


In [ ]:
config["scenario_run"]["threads"] = 8
run_sfincs_solver = True

display(pd.Series({
    "dynamic_wflow_handoff": "accepted",
    "sfincs_solver": "run" if run_sfincs_solver else "stage forcing only",
    "sfincs_threads": config["scenario_run"]["threads"],
}, name="interactive_model_run"))

## 4 - Pre-run forcing QA


In [ ]:
forcing_stage = stage_inland_coupled_example_forcing(
    config,
    {"location_root": location_root},
    example_plan=example_plan,
    event_id=event_id,
    force=rerun,
)

scenario_report = forcing_stage.scenario_report
sfincs_report = forcing_stage.sfincs_report
sim = forcing_stage.sim

display(scenario_report[["event_id", "sfincs_domain_id", "scenario_status", "run_root", "wflow_discharge_forcing"]])
display(sfincs_report[["event_id", "sfincs_domain_id", "status", "n_src", "run_root", "wflow_discharge_forcing"]])
for _, row in sfincs_report.iterrows():
    print(f"{row['sfincs_domain_id']}: staged forcing written; run pre-run QA before launching SFINCS.")

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

for domain_id, s in sim.items():
    run_dir = s["run_dir"]
    manifest_path = run_dir / "forcing_manifest.json"
    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.exists() else {}

    def _run_path(value):
        if value in (None, ""):
            return None
        path = Path(str(value))
        if path.is_absolute():
            return path
        return run_dir / path if (run_dir / path).exists() else location_root / path

    discharge_path = _run_path(manifest.get("wflow_discharge_forcing"))
    precip_path = _run_path(manifest.get("prepared_precip") or manifest.get("direct_rainfall_source"))
    event_label = f"{event_id} · {domain_id}"

    display(pd.Series({
        "event": event_label,
        "run_root": str(run_dir),
        "forcing_mode": manifest.get("forcing_mode"),
        "direct_rainfall_enabled": manifest.get("direct_rainfall_enabled"),
        "wflow_discharge_forcing": None if discharge_path is None else str(discharge_path),
        "rainfall_member_id": manifest.get("rainfall_member_id"),
        "soil_moisture_member_id": manifest.get("soil_moisture_member_id"),
    }, name="inland_forcing_summary"))

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)
    axes = axes.ravel()

    if discharge_path is not None and discharge_path.exists():
        with xr.open_dataset(discharge_path) as ds:
            discharge = ds["discharge"].load()
        if "time" in discharge.dims:
            source_dims = [dim for dim in discharge.dims if dim != "time"]
            total = discharge.sum(source_dims, skipna=True) if source_dims else discharge
            total.to_pandas().plot(ax=axes[0], color="#08519c", linewidth=1.8)
            axes[0].set_title("Wflow discharge handoff to SFINCS")
            axes[0].set_ylabel("discharge [m3/s]")
        else:
            values = np.asarray(discharge.values, dtype=float).ravel()
            axes[0].hist(values[np.isfinite(values)], bins=40, color="#3182bd", alpha=0.75)
            axes[0].set_title("Wflow discharge handoff distribution")
    else:
        axes[0].text(0.5, 0.5, "No sfincs_discharge.nc yet\nrun Wflow replay first", ha="center", va="center", transform=axes[0].transAxes)
        axes[0].set_title("Wflow discharge handoff to SFINCS")

    if precip_path is not None and precip_path.exists():
        with xr.open_dataset(precip_path) as ds:
            var = "precip" if "precip" in ds else next(iter(ds.data_vars))
            rain = ds[var].load()
        if "time" in rain.dims:
            dims = [dim for dim in rain.dims if dim != "time"]
            rain.mean(dims, skipna=True).to_pandas().plot(ax=axes[1], color="#08519c", linewidth=1.8)
            axes[1].set_ylabel("precipitation [mm/hr]")
        else:
            rain.plot(ax=axes[1], cmap="Blues", cbar_kwargs=dict(shrink=0.75, label="rainfall"))
        axes[1].set_title("Rainfall hyetograph")
    else:
        axes[1].text(0.5, 0.5, "No precipitation forcing staged", ha="center", va="center", transform=axes[1].transAxes)
        axes[1].set_title("Rainfall hyetograph")

    smax = np.fromfile(run_dir / "sfincs.smax", dtype="<f4") if (run_dir / "sfincs.smax").exists() else np.array([])
    seff = np.fromfile(run_dir / "sfincs.seff", dtype="<f4") if (run_dir / "sfincs.seff").exists() else np.array([])
    if smax.size and seff.size:
        valid = np.isfinite(smax) & np.isfinite(seff) & (smax > 0)
        frac = seff[valid] / smax[valid] if valid.any() else np.array([])
        axes[2].hist(frac, bins=40, color="#6baed6", edgecolor="white", linewidth=0.4)
        if frac.size:
            axes[2].axvline(float(np.median(frac)), color="#08519c", linestyle="--", label=f"median={float(np.median(frac)):.2f}")
            axes[2].legend(fontsize=8)
        axes[2].set_xlabel("seff / smax")
    else:
        axes[2].text(0.5, 0.5, "No smax/seff files staged", ha="center", va="center", transform=axes[2].transAxes)
    axes[2].set_title("Initial SFINCS soil saturation")

    ks = np.fromfile(run_dir / "sfincs.ks", dtype="<f4") if (run_dir / "sfincs.ks").exists() else np.array([])
    if ks.size:
        finite_ks = ks[np.isfinite(ks) & (ks > 0)]
        axes[3].hist(finite_ks, bins=50, color="#8c6d31", alpha=0.75)
        axes[3].set_xlabel("mm/hr")
        if finite_ks.size:
            p50, p95 = np.percentile(finite_ks, [50, 95])
            axes[3].set_title(f"Ksat (p50={p50:.1f}, p95={p95:.1f})")
        else:
            axes[3].set_title("Infiltration hydraulic conductivity")
    else:
        axes[3].text(0.5, 0.5, "No sfincs.ks file staged", ha="center", va="center", transform=axes[3].transAxes)
        axes[3].set_title("Infiltration hydraulic conductivity")

    for ax in axes:
        ax.grid(True, alpha=0.25)
    out_path = run_dir / "diagnostics" / f"{event_id}_inland_forcing_qa.png"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=160)
    plt.show()
    print("Saved inland forcing QA plot:", out_path)

if not sim:
    print("No example runs — nothing to QA.")

## 5 - Wflow handoff output

Review the dynamic Wflow response that is handed to SFINCS: source peak discharge, staged rainfall total when direct rainfall is enabled, and handoff hydrographs.


## 4 - Pre-run forcing QA


## 5 - Run SFINCS


In [ ]:
if run_sfincs_solver:
    sfincs_rows = []
    for domain_id, s in sim.items():
        run_dir = s["run_dir"]
        result = run_model(run_dir, config=config)
        s["result"] = result
        manifest_path = run_dir / "forcing_manifest.json"
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        manifest["sfincs_run_executed"] = True
        manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
        sfincs_rows.append({
            "event_id": event_id,
            "sfincs_domain_id": domain_id,
            "status": "completed",
            "n_src": int(s["dis"].sizes.get("index", 0)),
            "run_root": str(run_dir),
            "map_written": result.map_path.exists(),
        })
    sfincs_run_report = pd.DataFrame(sfincs_rows)
    display(sfincs_run_report[["event_id", "sfincs_domain_id", "status", "n_src", "run_root", "map_written"]])
else:
    print("SFINCS solver skipped; staged forcing and QA are available for review.")

In [ ]:
import numpy as np
import matplotlib.animation as animation
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import xarray as xr

for domain_id, s in sim.items():
    run_dir = s["run_dir"]
    discharge_df = s["dis"].transpose("time", "index").to_pandas()
    disch = discharge_df.sum(axis=1)
    if not isinstance(disch.index, pd.DatetimeIndex):
        disch = pd.Series(disch.to_numpy(), index=pd.date_range(s["t_start"], periods=len(disch), freq="h"))

    with xr.open_dataset(run_dir / "sfincs_map.nc", decode_times=False) as ds:
        x = ds["x"].values.astype(float)
        y = ds["y"].values.astype(float)
        time_s = ds["time"].values.astype(float)
        zb = ds["zb"].values.astype(float)
        msk = ds["msk"].values.astype(float)
        zs = ds["zs"].values.astype(float)

    timestamps = [pd.Timestamp(s["t_start"]) + pd.Timedelta(seconds=float(t)) for t in time_s]
    active = np.isfinite(msk) & (msk > 0)
    depth = np.where(active[None, :, :] & ((zs - zb[None, :, :]) > 0.05), zs - zb[None, :, :], np.nan)
    peak_depth = np.nanmax(depth, axis=0)
    depth_values = peak_depth[np.isfinite(peak_depth)]
    depth_vmax = max(float(np.nanpercentile(depth_values, 99)) if depth_values.size else 0.5, 0.5)
    wet_km2 = np.array([float(np.isfinite(depth[i]).sum() * 90 * 90 / 1e6) for i in range(len(timestamps))])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
    disch.plot(ax=axes[0], color="#08519c", linewidth=2, label="total inflow")
    peak_i = int(np.nanargmax(wet_km2))
    axes[0].axvline(timestamps[peak_i], color="#444444", linestyle=":", linewidth=1.5, label="peak flood time")
    axes[0].set_title(f"{event_id} · {domain_id} Wflow-to-SFINCS inflow")
    axes[0].set_ylabel("Discharge [m3/s]")
    axes[0].grid(True, alpha=0.25)
    axes[0].legend()
    mesh = axes[1].pcolormesh(x, y, np.ma.masked_invalid(peak_depth), cmap="Blues", norm=mcolors.Normalize(0, depth_vmax), shading="auto")
    axes[1].set_aspect("equal")
    axes[1].set_title(f"Peak flood depth ({timestamps[peak_i]:%Y-%m-%d %H:%M})")
    fig.colorbar(mesh, ax=axes[1], shrink=0.8, label="flood depth [m]")
    plt.show()

    figures_dir = run_dir / "figures"
    figures_dir.mkdir(parents=True, exist_ok=True)
    out_mp4 = figures_dir / f"{event_id}_flood_discharge_animation.mp4"
    fig_anim = plt.figure(figsize=(10, 10), facecolor="#1a1a2e")
    grid = gridspec.GridSpec(2, 1, height_ratios=[4, 1], hspace=0.18, figure=fig_anim)
    ax_map = fig_anim.add_subplot(grid[0])
    ax_hyd = fig_anim.add_subplot(grid[1])
    ax_map.set_facecolor("#1a1a2e")
    ax_hyd.set_facecolor("#1a1a2e")
    flood_mesh = ax_map.pcolormesh(x, y, np.ma.masked_invalid(depth[0]), cmap="Blues", norm=mcolors.Normalize(0, depth_vmax), shading="auto", alpha=0.82)
    title_txt = ax_map.set_title("", color="white", fontsize=10)
    ax_map.set_aspect("equal")
    ax_map.tick_params(colors="white")
    ax_hyd.plot(disch.index, disch.to_numpy(), color="#6baed6", linewidth=1.8)
    cursor = ax_hyd.axvline(timestamps[0], color="#fd8d3c", linewidth=1.6)
    ax_hyd.tick_params(colors="white", labelsize=8)
    ax_hyd.set_ylabel("Inflow [m3/s]", color="white")
    ax_hyd.grid(True, alpha=0.2)

    def _update(i):
        flood_mesh.set_array(np.ma.masked_invalid(depth[i]).ravel())
        cursor.set_xdata([timestamps[i], timestamps[i]])
        title_txt.set_text(f"{event_id} · {domain_id} | {timestamps[i]:%Y-%m-%d %H:%M} | flooded={wet_km2[i]:.2f} km2")
        return flood_mesh, cursor, title_txt

    ani = animation.FuncAnimation(fig_anim, _update, frames=len(timestamps), interval=120, blit=False)
    ani.save(str(out_mp4), writer=animation.FFMpegWriter(fps=10, bitrate=1600, codec="libx264", extra_args=["-pix_fmt", "yuv420p"]), dpi=140)
    plt.close(fig_anim)
    display(HTML(f"<h4>{event_id} · {domain_id} — flood + discharge animation</h4>"))
    display(Video(str(out_mp4), embed=True))

if not sim:
    print("No example runs — nothing to animate.")

## 6 - Flood + discharge animation (mp4)


## 7 - Post-run diagnostics

Review run diagnostics and outputs after the example completes.
